In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!cp -r /kaggle/input/notebooks/""/sonuc-analizi-retriever/chroma_db .
!cp -r /kaggle/input/notebooks/""/sonuc-analizi-retriever/chroma_db_contextual .
!cp  /kaggle/input/notebooks/""/sonuc-analizi-retriever/testset_mixed_102_soru_20260829_060905.csv .
!cp  /kaggle/input/notebooks/""/sonuc-analizi-retriever/chunk_map_20260829_060905.csv .

In [ ]:
# ── ADIM 1: GEREKLİ KÜTÜPHANELERİ YÜKLE ──
!pip install -q "langchain-core>=0.3.0,<0.4.0" "langchain>=0.3.0,<0.4.0" "langchain-community>=0.3.0,<0.4.0" langchain-chroma langchain-ollama langchain-huggingface ragas

print('Kütüphaneler başarıyla yüklendi.')

In [ ]:
!apt-get update && apt-get install -y zstd || sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# ── ADIM 2: YEREL OLLAMA SERVİSİNİ KUR VE BAŞLAT (GPU 0) ──
import os
import subprocess
import time
import shutil

# 1. Ollama kurulu değilse indir ve kur
if not shutil.which("ollama"):
    print("Ollama yerel sistemde bulunamadı. Kurulum başlatılıyor (curl -fsSL https://ollama.com/install.sh | sh)...")
    os.system("curl -fsSL https://ollama.com/install.sh | sh")
    time.sleep(3)
else:
    print("[✓] Ollama sistemde zaten kurulu.")

print("Yerel Ollama servisi başlatılıyor (Port: 11434)...")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

# Arka planda ollama serve başlat
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Modeli pull et (kontrol et/indir)
print("Gemma modeli indiriliyor/kontrol ediliyor (gemma4:e4b)...")
subprocess.run(["ollama", "pull", "llama3.1:8b"], stdout=subprocess.DEVNULL)
print("[✓] Yerel Ollama servisi ve llama3.1:8b  modeli hazır!")

In [ ]:
!pip install rank_bm25
!pip install sentence-transformers

In [ ]:
!mkdir outputs

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

llama_model = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0.0,
    num_ctx=16384
)

In [ ]:
import os
import sys
import datetime
import time
import asyncio
import hashlib
import pandas as pd

from openai import OpenAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.messages import BaseMessage, AIMessage

from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter


import shutil
import pandas as pd
from langchain_core.documents import Document
from langchain_chroma import Chroma

import ast
import time
import asyncio
import pandas as pd
from openai import OpenAI
from datasets import Dataset


In [ ]:
# 2. Dosya Yolları ve Veri Setleri
chunk_map_path = "/kaggle/working/chunk_map_20260829_060905.csv"
testset_path = "/kaggle/working/testset_mixed_102_soru_20260829_060905.csv"
db_dir = "/kaggle/working/chroma_db"
contextual_db_dir = "/kaggle/working/chroma_db_contextual"

df_map = pd.read_csv(chunk_map_path)
df_test = pd.read_csv(testset_path)

# Test setindeki sütunları kontrol edelim (Doğru ground truth sütununu yakalamak için)
print(f"[*] Test seti sütunları: {df_test.columns.tolist()}")


In [ ]:
docs_for_bm25 = []
for _, row in df_map.iterrows():
    clean_text = str(row['text']).strip()
    chunk_id = str(row['chunk_id'])
    docs_for_bm25.append(Document(page_content=clean_text, metadata={"chunk_id": chunk_id}))

# 3. Embedding ve Veritabanı Bağlantıları
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

db = Chroma(persist_directory=db_dir, embedding_function=embeddings)
db_contextual = Chroma(persist_directory=contextual_db_dir, embedding_function=embeddings)

contextual_data = db_contextual.get()
docs_for_ctx_bm25 = []
for doc_text, metadata in zip(contextual_data['documents'], contextual_data['metadatas']):
    docs_for_ctx_bm25.append(Document(page_content=doc_text, metadata={"chunk_id": str(metadata.get('chunk_id'))}))

In [ ]:
from langchain_community.retrievers import BM25Retriever
import pandas as pd
import ast
import numpy as np
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# Reranker modülleri
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers import ContextualCompressionRetriever

K = 5
INITIAL_K_FOR_RERANK = 15


basic_retriever = db.as_retriever(search_kwargs={"k": K})

# Normal Hibrit (Contextual Olmayan: Standart BM25 + Standart Vektör)
bm25_std = BM25Retriever.from_documents(docs_for_bm25)
bm25_std.k = K
vector_std = db.as_retriever(search_kwargs={"k": K})
normal_hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_std, vector_std], weights=[0.5, 0.5]
)

# Contextual + Hibrit (Saf)
bm25_ctx = BM25Retriever.from_documents(docs_for_ctx_bm25)
bm25_ctx.k = K
contextual_chroma_std = db_contextual.as_retriever(search_kwargs={"k": K})
contextual_hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_ctx, contextual_chroma_std], weights=[0.5, 0.5]
)

# Ctx + Hibrit + Reranker (Saf)
bm25_ctx_wide = BM25Retriever.from_documents(docs_for_ctx_bm25)
bm25_ctx_wide.k = INITIAL_K_FOR_RERANK
contextual_wide_dense = db_contextual.as_retriever(search_kwargs={"k": INITIAL_K_FOR_RERANK})
contextual_hybrid_wide = EnsembleRetriever(
    retrievers=[bm25_ctx_wide, contextual_wide_dense], weights=[0.5, 0.5]
)

reranker_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3", model_kwargs={'device': 'cuda'})
compressor = CrossEncoderReranker(model=reranker_model, top_n=K)
contextual_reranker_pure = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=contextual_hybrid_wide)

retrievers_to_test = {
    "1. Temel Vektör": basic_retriever,
    "2. Normal Hibrit (Contextual Yok)": normal_hybrid_retriever,
    "3. Contextual + Hibrit (Saf)": contextual_hybrid_retriever,
    "4. Ctx + Hibrit + Reranker (Saf)": contextual_reranker_pure
}

In [ ]:
eval_data = []
total_soru = len(df_test)
print(f"\n[*] Toplam {total_soru} soru işlenecek. İlerleme her 10 soruda bir raporlanacaktır.\n" + "-"*60)

for system_name, retriever in retrievers_to_test.items():
    print(f"\n🔄 [SİSTEM BAŞLADI] Mimari: {system_name}")
    print(f"   -> 102 soru için Ollama (Llama 3.1) yanıtları üretiliyor, lütfen bekleyin...")
    
    count = 0
    for index, row in df_test.iterrows():
        query = row['user_input']
        
        # RAGAS ground truth (ideal cevap metni) sütununu güvenli bir şekilde alıyoruz
        ground_truth_answer = ""
        for gt_col in ['ground_truth', 'response', 'answer', 'ideal_answer']:
            if gt_col in df_test.columns and pd.notna(row[gt_col]):
                ground_truth_answer = str(row[gt_col])
                break
                
        # Eğer yukarıdakiler bulunamazsa fallback olarak reference_contexts stringleştirilir
        if not ground_truth_answer and 'reference_contexts' in df_test.columns:
            try:
                gt_list = ast.literal_eval(row['reference_contexts'])
                ground_truth_answer = " ".join(gt_list)
            except Exception:
                ground_truth_answer = str(row['reference_contexts'])
                
        if not ground_truth_answer:
            continue
            
        # A) Retrieval
        retrieved_docs = retriever.invoke(query)
        contexts = [doc.page_content for doc in retrieved_docs[:K]]
        if not contexts:
            continue
            
        # B) Generation (Ollama - Llama 3.1)
        context_text = "\n\n---\n\n".join(contexts)
        prompt_text = f"""You are a helpful AI assistant. Use the following context to answer the user's question accurately. If you don't know based on the context, say that you don't know.

Context:
{context_text}

Question: {query}
Answer:"""
                
        response_msg = llama_model.invoke(prompt_text)
        
        # ChatOllama doğrudan .content döner
        answer = response_msg.content if hasattr(response_msg, "content") else str(response_msg)
        
        eval_data.append({
            "system_name": system_name,
            "question": query,
            "answer": answer,
            "contexts": contexts,
            "ground_truth": ground_truth_answer # RAGAS'ın beklediği tekil metin formatı
        })
        count += 1
        
        # --- İlerleme Takibi İçin Eklenen Print ---
        if count % 10 == 0:
            df_eval_raw = pd.DataFrame(eval_data)
            df_eval_raw.to_csv("tahminler_opensource_llama.csv", index=False)
            print("\n[*] Tüm süreç tamamlandı! 'tahminler.csv' dosyasına kaydedildi.")
            print(f"   ... {count}/{total_soru} soru işlendi.")
                        
    print(f"   [✓] {system_name} için {count} soru başarıyla tamamlandı.")

df_eval_raw = pd.DataFrame(eval_data)
df_eval_raw.to_csv("tahminler_opensource_llama.csv", index=False)
print("\n[*] Tüm süreç tamamlandı! 'tahminler.csv' dosyasına kaydedildi.")